# Hera Dynamic Toolkits – Quickstart

This notebook shows how to **register**, **discover**, and **load** toolkits dynamically using Hera's repository mechanism.

It covers two ways:
1. Via the provided **CLI** (`hera.utils.data.cli_toolkit_repository`)
2. Via a small **Python API** (`ToolkitHome.registerToolkit`)

> You can use this notebook as a template for any external experiment/toolkit you keep on disk.


## 0) Prerequisites
- You have a working Hera environment/venv.
- Your project name (we'll use `UnitTestProject` by default).
- Your external toolkit lives on disk and exposes a class you want to load (e.g., `MyPkg.my_module.MyToolkit`).


In [ ]:

# Edit this to your project
PROJECT = "UnitTestProject"
print("Using project:", PROJECT)


## 1) (Option A) Register a toolkit via CLI
The CLI creates a **ToolkitDataSource** document that describes:
- `resource` – the directory on disk that contains the module file
- `desc.classpath` – the Python class to import (e.g., `mypkg.mymodule.MyToolkit`)
- `desc.parameters` – keyword-args for class `__init__`

**Example:**

In [ ]:

        # This cell SHOWS the command; run it in a terminal if you prefer CLI.
        # Replace the placeholders with your values.
        print(
            "python -m hera.utils.data.cli_toolkit_repository register-datasource \n"
            "  --project {project} \
"
            "  --name DemoToolkit_DS \
"
            "  --classpath demo_pkg.demo_module.DemoToolkit \
"
            "  --resource /path/to/demo_pkg \
"
            "  --params '{"alpha": 7}' \
"
            "  --version 0.0.1 \
"
            "  --overwrite".format(project=PROJECT)
        )


## 1) (Option B) Register a toolkit via Python
If you prefer Python code, you can pass the **class object** directly. The system will inspect it, derive the source directory, and write the DB document.

In [ ]:

import sys, os, importlib
from hera.toolkit import ToolkitHome

# Example only: make your package importable (if not already installed):
# sys.path.insert(0, "/path/to")  # parent folder that contains 'demo_pkg'
# DemoToolkit = getattr(importlib.import_module("demo_pkg.demo_module"), "DemoToolkit")

# th = ToolkitHome()
# doc = th.registerToolkit(
#     toolkitclass=DemoToolkit,
#     projectName=PROJECT,
#     datasource_name="DemoToolkit_DS",
#     params={"alpha": 7},
#     version=(0, 0, 1),
#     overwrite=True,
# )
# print("Registered:", doc)
print("Fill the example above with your real module/class to register dynamically.")


## 2) Discover what toolkits are available
Static (built-in) toolkits **and** dynamic (from DB) are shown together:

In [ ]:

from hera.toolkit import ToolkitHome
th = ToolkitHome()
df = th.getToolkitTable(PROJECT)
print("Toolkits table shape:", df.shape)
df.head(20)


## 3) Load a toolkit class dynamically
You can load directly from the saved document via `DataHandler_Class.getData` or use the table to locate your data source.

In [ ]:

from hera.datalayer import Project
from hera.datalayer.datahandler import DataHandler_Class

p = Project(projectName=PROJECT)

# Find your datasource document by name (the one you registered)
docs = p.getMeasurementsDocuments(type="ToolkitDataSource", datasourceName="DemoToolkit_DS")
if not docs:
    print("No document named 'DemoToolkit_DS' – register first (CLI or Python).")
else:
    doc = docs[0]
    obj = DataHandler_Class.getData(resource=doc.resource, desc=doc.desc)
    print("Loaded class:", type(obj).__name__, "| resource:", doc.resource)
    # Now you can call your class' methods...


---
**That’s it!**

Use this template for your own external toolkits/experiments. Register → Discover → Load → Use.